# Publication Outputs

Generates manuscript-ready figures (300 DPI PNG + vector PDF) and tables (CSV + Markdown + LaTeX, with captions/abbreviations/notes) from the project's already-computed results.

**Data source policy:** this notebook reads only files already saved under `outputs/tables/` by earlier notebooks (01-05), including `test_predictions_{model}.csv` — raw per-case test predictions saved as a byproduct of `05_final_evaluation.ipynb`'s single authorized test-set evaluation. **No model or calibrator is refit here, no prediction is recomputed, and no value is simulated or fabricated** — every number in every figure and table traces back to a saved file.

**Prerequisite:** notebooks 01-05 must have been run first (against your own local MIMIC-IV extract) so that `outputs/tables/` and `outputs/models/` are populated.

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path

from src.config import load_config
from src.reproducibility import set_global_seed, get_logger, log_run_metadata
from src.plot_publication_figures import (
    generate_all_publication_figures,
    export_all_publication_tables,
    load_all_test_predictions,
)

config = load_config()
seed = config["project"]["random_seed"]
set_global_seed(seed)
logger = get_logger(log_file=config["logging"]["log_file"])
logger.info("Publication outputs notebook started (seed=%d)", seed)

## 1. Load saved test predictions

Raises `FileNotFoundError` with a clear message if `05_final_evaluation.ipynb` has not been run yet — this notebook does not compute predictions itself.

In [ ]:
test_predictions = load_all_test_predictions(config)
for model_name, df in test_predictions.items():
    print(f"{model_name}: {len(df)} test cases, calibration_method={df['calibration_method'].iloc[0]}")

## 2. Generate all figures

1. Study flowchart (Figure 1) — from `outputs/tables/cohort_flow_counts.csv`
2. Proposed system architecture (Figure 2) — static, grounded in actual module names
3. Reliability diagrams (Figure 3) — one per model, calibrated test predictions
4. ROC curves (Figure 4) — all models overlaid
5. Precision-recall curves (Figure 5) — all models overlaid
6. Risk-coverage curve (Figure 6) — reuses `src.selective_prediction.evaluate_referral_policy` on test predictions
7. Subgroup calibration comparison (Figure 7) — reliability curves split by subgroup

Each figure is saved as both a 300 DPI PNG and a vector PDF. Mermaid source files are generated for Figures 1 and 2.

In [ ]:
figure_outputs = generate_all_publication_figures(config)

for name, paths in figure_outputs.items():
    logger.info("Generated %s -> %s", name, paths)
    print(name, "->", paths)

## 3. Export all tables (CSV + Markdown + LaTeX, with captions/abbreviations/notes)

Tables 1-6. Every note is populated from the actual saved table or run-metadata JSON (sample sizes, bootstrap count, seed) — never a hardcoded illustrative number.

In [ ]:
table_exports = export_all_publication_tables(config)

for table_name, paths in table_exports.items():
    logger.info("Exported %s -> %s", table_name, paths)
    print(table_name, "->", {fmt: str(p) for fmt, p in paths.items()})

## 4. Preview: Table 3 (main results) rendered as Markdown

In [ ]:
if "table3_main_results" in table_exports:
    from IPython.display import Markdown, display
    md_text = table_exports["table3_main_results"]["md"].read_text(encoding="utf-8")
    display(Markdown(md_text))
else:
    print("table_3_main_results.csv not found — run 05_final_evaluation.ipynb first.")

## 5. Run metadata

In [ ]:
log_run_metadata(
    output_path=config["publication"]["run_metadata_path"],
    seed=seed,
    extra={
        "step": "publication_outputs",
        "figures_generated": list(figure_outputs.keys()),
        "tables_exported": list(table_exports.keys()),
        "data_source": "outputs/tables/ only — no predictions recomputed, no values fabricated",
    },
)
logger.info("Publication outputs notebook finished")
print(f"Figures saved to: {config['publication']['figures_output_dir']}")
print(f"Tables saved to: {config['publication']['tables_output_dir']}")